In [3]:
import cv2
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from email.mime.application import MIMEApplication
from ultralytics import YOLO
import time
import os
from collections import defaultdict
import numpy as np

# Email configuration
EMAIL_SENDER = "chethanks407@gmail.com"
EMAIL_PASSWORD = "pzco ftmv gbxq swqv"  # Use app-specific password for Gmail
EMAIL_RECEIVER = "chethanks506@gmail.com"
SMTP_SERVER = "smtp.gmail.com"
SMTP_PORT = 587

# Alert configuration
ALERT_INTERVAL = 10  # Minimum seconds between alerts for the same object class
MIN_CONFIDENCE = 0.5  # Minimum confidence threshold for alerts

class UniversalObjectDetector:
    def __init__(self):
        self.model = YOLO('yolov8n.pt')  # or yolov8s/m/l/x for better accuracy
        self.last_alert_times = defaultdict(float)
        self.cap = None
        self.temp_dir = "temp_detections"
        os.makedirs(self.temp_dir, exist_ok=True)
        
        # Common obstacle classes in COCO dataset (YOLOv8 default classes)
        self.obstacle_classes = [
            'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 
            'traffic light', 'stop sign', 'parking meter', 'bench',
            'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe',
            'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
            'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard',
            'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl',
            'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
            'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet',
            'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven',
            'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
            'hair drier', 'toothbrush'
        ]

    def send_email_alert(self, detections, frame):
        try:
            # Create email message
            msg = MIMEMultipart()
            msg['From'] = EMAIL_SENDER
            msg['To'] = EMAIL_RECEIVER
            msg['Subject'] = f"Object Detection Alert: {len(detections)} objects detected"

            # Email body with detection details
            body = "The following objects were detected:\n\n"
            for class_name, count in detections.items():
                body += f"- {class_name}: {count} detected\n"
            
            body += f"\nTimestamp: {time.strftime('%Y-%m-%d %H:%M:%S')}"
            msg.attach(MIMEText(body, 'plain'))

            # Attach the detection image
            timestamp = int(time.time())
            image_filename = os.path.join(self.temp_dir, f"detection_{timestamp}.jpg")
            cv2.imwrite(image_filename, frame)
            
            with open(image_filename, 'rb') as f:
                img_data = f.read()
            image_attachment = MIMEImage(img_data, name=os.path.basename(image_filename))
            msg.attach(image_attachment)

            # Send email
            with smtplib.SMTP(SMTP_SERVER, SMTP_PORT) as server:
                server.starttls()
                server.login(EMAIL_SENDER, EMAIL_PASSWORD)
                server.send_message(msg)
            
            print(f"Email alert sent successfully at {time.strftime('%Y-%m-%d %H:%M:%S')}")
            
            # Clean up temporary file
            os.remove(image_filename)
            
        except Exception as e:
            print(f"Error sending email: {str(e)}")

    def process_detections(self, results):
        """Process YOLO detection results and return detected objects"""
        detections = defaultdict(int)
        obstacle_detected = False
        
        for result in results:
            if result.boxes is not None:
                for box in result.boxes:
                    confidence = box.conf.item()
                    class_id = int(box.cls.item())
                    class_name = self.model.names[class_id]
                    
                    if confidence >= MIN_CONFIDENCE and class_name in self.obstacle_classes:
                        detections[class_name] += 1
                        obstacle_detected = True
        
        return detections, obstacle_detected

    def draw_detections(self, frame, results):
        """Draw bounding boxes and labels on the frame"""
        for result in results:
            if result.boxes is not None:
                for box in result.boxes:
                    confidence = box.conf.item()
                    class_id = int(box.cls.item())
                    class_name = self.model.names[class_id]
                    
                    if confidence >= MIN_CONFIDENCE and class_name in self.obstacle_classes:
                        x1, y1, x2, y2 = map(int, box.xyxy[0])
                        
                        # Draw bounding box
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                        
                        # Draw label with confidence
                        label = f"{class_name}: {confidence:.2f}"
                        cv2.putText(frame, label, (x1, y1 - 10), 
                                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        return frame

    def should_send_alert(self, detections):
        """Check if alert should be sent based on time intervals"""
        current_time = time.time()
        should_alert = False
        
        for class_name in detections.keys():
            if current_time - self.last_alert_times[class_name] >= ALERT_INTERVAL:
                should_alert = True
                self.last_alert_times[class_name] = current_time
        
        return should_alert

    def start_detection(self, camera_index=0):
        """Start obstacle detection from camera feed"""
        self.cap = cv2.VideoCapture(camera_index)
        
        if not self.cap.isOpened():
            print("Error: Could not open camera")
            return
        
        print("Starting obstacle detection. Press 'q' to quit.")
        
        try:
            while True:
                ret, frame = self.cap.read()
                if not ret:
                    print("Error: Failed to capture frame")
                    break
                
                # Perform object detection
                results = self.model(frame, verbose=False)
                
                # Process detections
                detections, obstacle_detected = self.process_detections(results)
                
                # Draw detections on frame
                frame_with_detections = self.draw_detections(frame.copy(), results)
                
                # Display frame with detections
                cv2.imshow('Obstacle Detection', frame_with_detections)
                
                # Check if alert should be sent
                if obstacle_detected and detections and self.should_send_alert(detections):
                    print(f"Obstacles detected: {dict(detections)}")
                    self.send_email_alert(detections, frame_with_detections)
                
                # Check for quit command
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                
                # Add small delay to reduce CPU usage
                time.sleep(0.1)
                
        except KeyboardInterrupt:
            print("Detection stopped by user")
        finally:
            self.cap.release()
            cv2.destroyAllWindows()

    def detect_from_image(self, image_path):
        """Detect obstacles from a single image"""
        if not os.path.exists(image_path):
            print(f"Error: Image file {image_path} not found")
            return
        
        frame = cv2.imread(image_path)
        if frame is None:
            print("Error: Could not read image")
            return
        
        # Perform object detection
        results = self.model(frame, verbose=False)
        
        # Process detections
        detections, obstacle_detected = self.process_detections(results)
        
        # Draw detections on frame
        frame_with_detections = self.draw_detections(frame.copy(), results)
        
        # Display results
        print(f"Detected objects: {dict(detections)}")
        
        # Show image with detections
        cv2.imshow('Obstacle Detection', frame_with_detections)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
        
        # Send alert if obstacles detected
        if obstacle_detected and detections:
            self.send_email_alert(detections, frame_with_detections)

# Main execution
if __name__ == "__main__":
    detector = UniversalObjectDetector()
    
    # Choose detection mode:
    # 1. Real-time detection from camera
    detector.start_detection(camera_index=0)  # Use default camera
    
    # 2. Or detect from an image file
    # detector.detect_from_image("path_to_your_image.jpg")

Starting obstacle detection. Press 'q' to quit.
Obstacles detected: {'person': 1}
Email alert sent successfully at 2025-09-13 18:56:28
Obstacles detected: {'person': 1}
Email alert sent successfully at 2025-09-13 18:56:38
Obstacles detected: {'cell phone': 1, 'person': 1}
Email alert sent successfully at 2025-09-13 18:56:43
Obstacles detected: {'person': 1, 'cell phone': 1}
Email alert sent successfully at 2025-09-13 18:56:48
